# Analyzing CliMA development throughput

How productive is CliMA's ecosystem, and how quickly does it turn pull requests
around? Here we track four quarterly metrics over time, both per repository and
across the whole ecosystem:

1. **Median time-to-merge (p50)** — the typical PR's wall-clock time from open
   to merge.
2. **90th-percentile time-to-merge (p90)** — the slow tail of PRs that sit
   unmerged for a long time.
3. **PR merge rate** — merged PRs per quarter.
4. **Release rate** — published releases per quarter.

All merged PRs are counted, including bot-authored ones (CompatHelper, TagBot,
Dependabot, GitHub Actions, etc.): automated dependency PRs that sit unmerged
for a long time are part of the resolution-speed picture we want to surface.

For the count-based panels (merge rate and release rate), the ecosystem-wide
("All repos") line is normalized by the total number of repositories, so it
reads as a per-repo average and stays on the same scale as the individual repo
traces.


In [1]:
%cd ..

/Users/pete/calkit/clima-perf


## Load merged pull requests

`fetch-github` stores merged PRs under `data/github/prs/<repo>/<YYYY-MM-DD>.json`
(one file per UTC day a PR was merged in that repo). Each record carries
`createdAt` and `mergedAt`, from which we derive the time-to-merge.


In [2]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import plotly.colors as pc
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT / "clima-perf" / "data").exists():
    ROOT = ROOT / "clima-perf"
DATA = ROOT / "data"
PRS_ROOT = DATA / "github" / "prs"

# Anything before 2018-Q1 predates the CliMA packaging effort; the time
# axis is cropped to that floor (matches the analyze notebook).
MIN_QUARTER = "2018-Q1"

# Minimum number of merged PRs in a repo-quarter for that point to be
# plotted; medians over only one or two PRs are too noisy to be useful.
MIN_PRS_PER_POINT = 3

# CamelCase repo lookup so legend labels match the canonical package names.
camel_by_lc: dict[str, str] = {}
creation_path = DATA / "github" / "repo_creation_dates.jsonl"
if creation_path.exists():
    for line in creation_path.read_text().splitlines():
        if line.strip():
            name = json.loads(line)["name"].replace(".jl", "")
            camel_by_lc[name.lower()] = name


def camel(name: str) -> str:
    return camel_by_lc.get(name.lower(), name)


def parse_iso(s: str) -> datetime:
    dt = datetime.fromisoformat(s.replace("Z", "+00:00"))
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def quarter_of(dt: datetime) -> str:
    return f"{dt.year}-Q{(dt.month - 1) // 3 + 1}"


def quarter_start(quarter: str) -> datetime:
    year, q = quarter.split("-Q")
    return datetime(int(year), (int(q) - 1) * 3 + 1, 1, tzinfo=timezone.utc)


rows: list[dict] = []
if PRS_ROOT.exists():
    for repo_dir in sorted(PRS_ROOT.iterdir()):
        if not repo_dir.is_dir():
            continue
        repo = camel(repo_dir.name)
        for f in repo_dir.glob("*.json"):
            for pr in json.loads(f.read_text()):
                author = pr.get("author") or {}
                created_raw = pr.get("createdAt")
                merged_raw = pr.get("mergedAt")
                if not created_raw or not merged_raw:
                    continue
                created = parse_iso(created_raw)
                merged = parse_iso(merged_raw)
                ttm_days = (merged - created).total_seconds() / 86400.0
                if ttm_days < 0:
                    continue
                quarter = quarter_of(merged)
                if quarter < MIN_QUARTER:
                    continue
                rows.append(
                    {
                        "repo": repo,
                        "number": pr.get("number"),
                        "author": author.get("login"),
                        "created_at": created_raw,
                        "merged_at": merged_raw,
                        "quarter": quarter,
                        "ttm_days": ttm_days,
                    }
                )

prs_df = pd.DataFrame(rows)
print(
    f"{len(prs_df)} merged PRs across "
    f"{prs_df['repo'].nunique() if not prs_df.empty else 0} repos"
)
prs_df.head()


12228 merged PRs across 22 repos


,repo,number,author,created_at,merged_at,quarter,ttm_days
0,CalibrateEmulateSample,85,jakebolewski,2020-11-18T00:57:16Z,2020-11-18T01:10:02Z,2020-Q4,0.008866
1,CalibrateEmulateSample,83,agarbuno,2020-11-03T03:29:19Z,2020-11-18T17:14:16Z,2020-Q4,15.572882
2,CalibrateEmulateSample,318,app/github-actions,2024-08-30T00:16:38Z,2024-09-07T20:37:13Z,2024-Q3,8.847627
3,CalibrateEmulateSample,223,odunbar,2023-07-12T17:28:38Z,2023-07-12T19:17:23Z,2023-Q3,0.075521
4,CalibrateEmulateSample,253,bielim,2023-11-13T16:00:22Z,2023-12-11T22:37:22Z,2023-Q4,28.275694


## Time-to-merge per quarter

We bin merged PRs by the quarter they were merged in and summarize the
time-to-merge with two statistics: the **median (p50)**, which tracks the
typical PR, and the **90th percentile (p90)**, which exposes the slow tail of
PRs that sit unmerged for a long time. One trace is drawn per repository (only
quarters with at least `MIN_PRS_PER_POINT` merged PRs are plotted), and a thick
gray line shows the statistic computed over *all* repositories' PRs combined.


In [3]:
empty = prs_df.empty

COLUMNS = ["repo", "quarter", "n_prs", "median_ttm_days", "p90_ttm_days"]


def summarize(grouped) -> pd.DataFrame:
    return grouped["ttm_days"].agg(
        n_prs="size",
        median_ttm_days="median",
        p90_ttm_days=lambda s: s.quantile(0.9),
    )


# Per-repo, per-quarter statistics + PR count.
if empty:
    repo_q = pd.DataFrame(columns=COLUMNS)
else:
    repo_q = summarize(prs_df.groupby(["repo", "quarter"])).reset_index()

# Ecosystem-wide statistics over all repos, per quarter.
if empty:
    all_q = pd.DataFrame(columns=COLUMNS)
else:
    all_q = summarize(prs_df.groupby("quarter")).reset_index()
    all_q.insert(0, "repo", "All repos")

ttm_df = pd.concat([repo_q, all_q], ignore_index=True)
ttm_df["quarter_start"] = ttm_df["quarter"].map(
    lambda q: quarter_start(q).date().isoformat()
)
ttm_df = (
    ttm_df[
        [
            "repo",
            "quarter",
            "quarter_start",
            "n_prs",
            "median_ttm_days",
            "p90_ttm_days",
        ]
    ]
    .sort_values(["repo", "quarter"])
    .reset_index(drop=True)
)

os.makedirs(ROOT / "results", exist_ok=True)
ttm_df.to_csv(ROOT / "results/pr-ttm.csv", index=False)
print(f"wrote {len(ttm_df)} rows to results/pr-ttm.csv")
ttm_df.head()


wrote 424 rows to results/pr-ttm.csv


,repo,quarter,quarter_start,n_prs,median_ttm_days,p90_ttm_days
0,All repos,2018-Q4,2018-10-01,5,0.004931,0.188382
1,All repos,2019-Q1,2019-01-01,27,0.170949,1.361903
2,All repos,2019-Q2,2019-04-01,71,0.139456,5.614525
3,All repos,2019-Q3,2019-07-01,115,0.173380,3.379785
4,All repos,2019-Q4,2019-10-01,181,0.104514,10.854468


## Releases per quarter

We also count published GitHub releases per quarter, per repository and across
the ecosystem, as a second productivity signal alongside the PR merge rate.


In [4]:
# fetch-github stores one release per line in
# data/github/releases/<repo>.jsonl. We count non-draft releases by the
# quarter they were published in, per repo and across the ecosystem.
RELEASES_ROOT = DATA / "github" / "releases"

rel_rows: list[dict] = []
if RELEASES_ROOT.exists():
    for f in sorted(RELEASES_ROOT.glob("*.jsonl")):
        repo = camel(f.stem)
        for line in f.read_text().splitlines():
            if not line.strip():
                continue
            rel = json.loads(line)
            if rel.get("draft"):
                continue
            published = rel.get("published_at") or rel.get("created_at")
            if not published:
                continue
            quarter = quarter_of(parse_iso(published))
            if quarter < MIN_QUARTER:
                continue
            rel_rows.append({"repo": repo, "quarter": quarter})

releases_df = pd.DataFrame(rel_rows)

if releases_df.empty:
    rel_repo_q = pd.DataFrame(columns=["repo", "quarter", "n_releases"])
    rel_all_q = pd.DataFrame(columns=["repo", "quarter", "n_releases"])
else:
    rel_repo_q = (
        releases_df.groupby(["repo", "quarter"])
        .size()
        .reset_index(name="n_releases")
    )
    rel_all_q = (
        releases_df.groupby("quarter").size().reset_index(name="n_releases")
    )
    rel_all_q.insert(0, "repo", "All repos")

n_rel_repos = releases_df["repo"].nunique() if not releases_df.empty else 0
print(f"{len(releases_df)} releases across {n_rel_repos} repos")
rel_repo_q.head()


1642 releases across 34 repos


,repo,quarter,n_releases
0,ArtifactWrappers,2021-Q2,2
1,ArtifactWrappers,2022-Q3,2
2,CalibrateEmulateSample,2022-Q2,1
3,CalibrateEmulateSample,2022-Q4,1
4,CalibrateEmulateSample,2023-Q3,1


In [5]:
# Repos ordered by total merged-PR volume (busiest first) for a stable,
# meaningful legend order; any release-only repos are appended at the end.
if empty:
    repo_order: list[str] = []
else:
    repo_order = prs_df["repo"].value_counts().index.tolist()

rel_repos = [] if releases_df.empty else rel_repo_q["repo"].unique().tolist()
repo_order = repo_order + [r for r in rel_repos if r not in repo_order]

palette = (
    list(pc.qualitative.Plotly)
    + list(pc.qualitative.D3)
    + list(pc.qualitative.Light24)
    + list(pc.qualitative.Dark24)
)
color_for = {
    repo: palette[i % len(palette)] for i, repo in enumerate(repo_order)
}

# The count-based panels (merge rate, releases) normalize the ecosystem-wide
# line by the total number of repos so it reads as a per-repo average and
# stays on the same scale as the individual repo traces.
n_pr_repos = prs_df["repo"].nunique() if not empty else 0
n_rel_repos = releases_df["repo"].nunique() if not releases_df.empty else 0

# Row 1 = median (p50), row 2 = p90, row 3 = PR merge rate, row 4 = releases.
# Each repo's traces share a legendgroup so toggling a repo hides it in every
# panel; the single legend entry lives on the merge-rate panel (row 3), which
# every repo with merged PRs appears in.
fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
)

TTM_METRICS = [
    (1, "median_ttm_days", "Median (p50)"),
    (2, "p90_ttm_days", "p90"),
]

for repo in repo_order:
    color = color_for[repo]

    # p50 / p90 panels: only quarters with enough merged PRs to be meaningful.
    sub = repo_q[
        (repo_q["repo"] == repo) & (repo_q["n_prs"] >= MIN_PRS_PER_POINT)
    ].sort_values("quarter")
    if not sub.empty:
        x = [quarter_start(q) for q in sub["quarter"]]
        for row, col, label in TTM_METRICS:
            fig.add_trace(
                go.Scatter(
                    x=x,
                    y=sub[col],
                    mode="lines+markers",
                    name=repo,
                    legendgroup=repo,
                    showlegend=False,
                    line=dict(color=color, width=1.5),
                    marker=dict(size=4),
                    customdata=sub[["quarter", "n_prs"]],
                    hovertemplate=(
                        f"<b>{repo}</b><br>"
                        "%{customdata[0]}<br>"
                        f"{label} TTM: %{{y:.1f}} days<br>"
                        "Merged PRs: %{customdata[1]}<extra></extra>"
                    ),
                ),
                row=row,
                col=1,
            )

    # Merge-rate panel: every quarter the repo had merged PRs. The repo's
    # legend entry lives here so even low-PR-volume repos are listed.
    rate = repo_q[repo_q["repo"] == repo].sort_values("quarter")
    if not rate.empty:
        fig.add_trace(
            go.Scatter(
                x=[quarter_start(q) for q in rate["quarter"]],
                y=rate["n_prs"],
                mode="lines+markers",
                name=repo,
                legendgroup=repo,
                showlegend=True,
                line=dict(color=color, width=1.5),
                marker=dict(size=4),
                customdata=rate[["quarter"]],
                hovertemplate=(
                    f"<b>{repo}</b><br>"
                    "%{customdata[0]}<br>"
                    "Merged PRs: %{y}<extra></extra>"
                ),
            ),
            row=3,
            col=1,
        )

    # Releases panel.
    rel = rel_repo_q[rel_repo_q["repo"] == repo].sort_values("quarter")
    if not rel.empty:
        fig.add_trace(
            go.Scatter(
                x=[quarter_start(q) for q in rel["quarter"]],
                y=rel["n_releases"],
                mode="lines+markers",
                name=repo,
                legendgroup=repo,
                showlegend=False,
                line=dict(color=color, width=1.5),
                marker=dict(size=4),
                customdata=rel[["quarter"]],
                hovertemplate=(
                    f"<b>{repo}</b><br>"
                    "%{customdata[0]}<br>"
                    "Releases: %{y}<extra></extra>"
                ),
            ),
            row=4,
            col=1,
        )

# Thick gray ecosystem-wide lines, added last so they sit on top. The TTM
# panels show the statistic over all PRs combined; the count panels show the
# per-repo average (total / number of repos).
if not all_q.empty:
    all_sorted = all_q.sort_values("quarter")
    all_x = [quarter_start(q) for q in all_sorted["quarter"]]
    for row, col, label in TTM_METRICS:
        fig.add_trace(
            go.Scatter(
                x=all_x,
                y=all_sorted[col],
                mode="lines",
                name="All repos",
                legendgroup="All repos",
                showlegend=False,
                line=dict(color="#666666", width=4),
                customdata=all_sorted[["quarter", "n_prs"]],
                hovertemplate=(
                    "<b>All repos</b><br>"
                    "%{customdata[0]}<br>"
                    f"{label} TTM: %{{y:.1f}} days<br>"
                    "Merged PRs: %{customdata[1]}<extra></extra>"
                ),
            ),
            row=row,
            col=1,
        )
    if n_pr_repos:
        fig.add_trace(
            go.Scatter(
                x=all_x,
                y=all_sorted["n_prs"] / n_pr_repos,
                mode="lines",
                name="All repos (aggregated)",
                legendgroup="All repos",
                showlegend=True,
                line=dict(color="#666666", width=4),
                customdata=all_sorted[["quarter", "n_prs"]],
                hovertemplate=(
                    "<b>All repos</b><br>"
                    "%{customdata[0]}<br>"
                    "Avg merged PRs/repo: %{y:.1f}<br>"
                    "Total merged PRs: %{customdata[1]}<extra></extra>"
                ),
            ),
            row=3,
            col=1,
        )

if not rel_all_q.empty and n_rel_repos:
    rel_all_sorted = rel_all_q.sort_values("quarter")
    fig.add_trace(
        go.Scatter(
            x=[quarter_start(q) for q in rel_all_sorted["quarter"]],
            y=rel_all_sorted["n_releases"] / n_rel_repos,
            mode="lines",
            name="All repos",
            legendgroup="All repos",
            showlegend=False,
            line=dict(color="#666666", width=4),
            customdata=rel_all_sorted[["quarter", "n_releases"]],
            hovertemplate=(
                "<b>All repos</b><br>"
                "%{customdata[0]}<br>"
                "Avg releases/repo: %{y:.2f}<br>"
                "Total releases: %{customdata[1]}<extra></extra>"
            ),
        ),
        row=4,
        col=1,
    )

fig.update_layout(
    title="Software development throughput",
    height=900,
    legend=dict(itemsizing="constant", font=dict(size=11)),
)
fig.update_yaxes(title_text="Median TTM (days)", row=1, col=1)
fig.update_yaxes(title_text="p90 TTM (days)", row=2, col=1)
fig.update_yaxes(title_text="PRs merged/quarter", row=3, col=1)
fig.update_yaxes(title_text="Releases/quarter", row=4, col=1)
fig.update_xaxes(title_text="Quarter", row=4, col=1)

os.makedirs(ROOT / "figures", exist_ok=True)
fig.write_json(ROOT / "figures/dev-throughput.json")
fig
